# Task 1.2 — Ingest Gate Events and Baggage Scans

- Gate events JSON: append to `bronze.gate_events`, partition by `event_date`
- Baggage scans CSV: 5-minute micro-batch append to `bronze.baggage_scans`
- DQ for gate events: `event_type` in known enum; `flight_id` must exist in flight schedule
- DQ for baggage: `scan_status` must be OK/EXCEPTION/MISREAD; bag_tag_number format validation
- Volume alert: if baggage scan count per hour < 5,000, trigger sensor health alert


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_1_2_Gate_Events_Baggage_Scans_Bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Part A — Gate Events Raw Ingestion

In [ ]:
# ── Raw Ingestion: gate_events.json (NDJSON) ─────────────────────
raw_gate_events_df = spark.read.json(
    f"{DATA_DIR}/gate_events/gate_events.json"
)
raw_gate_events_df = (raw_gate_events_df
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("event_date", F.to_date(F.col("event_ts")))
)

print(f"Raw gate events ingested: {raw_gate_events_df.count()}")
raw_gate_events_df.printSchema()
raw_gate_events_df.show(5, truncate=False)


## Part A — Gate Events Data Quality Checks

In [ ]:
# ── DQ Checks: Gate Events ────────────────────────────────────────
VALID_EVENT_TYPES = [
    "GATE_ASSIGNED", "GATE_OPEN", "BOARDING_START",
    "BOARDING_COMPLETE", "PUSHBACK", "GATE_CLOSED", "GATE_RELEASED"
]

# 1. event_type must be in known enum
invalid_event_type = raw_gate_events_df.filter(
    ~F.col("event_type").isin(VALID_EVENT_TYPES)
)
print(f"Invalid event_type records: {invalid_event_type.count()}")
if invalid_event_type.count() > 0:
    invalid_event_type.select("event_id", "event_type").show()

# 2. flight_id must exist in flight schedule
bronze_flight_df = spark.read.format("delta").load(f"{BRONZE_DIR}/flight_schedule")
valid_flight_ids = bronze_flight_df.select("flight_id").distinct()
orphan_events = (raw_gate_events_df
    .join(valid_flight_ids, "flight_id", "left_anti"))
print(f"Gate events with unknown flight_id: {orphan_events.count()}")

# Build validated dataframe
validated_gate_events_df = (raw_gate_events_df
    .filter(F.col("event_type").isin(VALID_EVENT_TYPES))
    .withColumn("event_ts", F.to_timestamp("event_ts"))
)
print(f"Validated gate events: {validated_gate_events_df.count()}")


## Part A — Write Gate Events to Bronze Delta

In [ ]:
# ── Write gate_events Bronze Delta (append, partitioned by event_date)
bronze_gate_path = f"{BRONZE_DIR}/gate_events"

(validated_gate_events_df.write
    .format("delta")
    .mode("append")
    .partitionBy("event_date")
    .save(bronze_gate_path))

print(f"Gate events written to {bronze_gate_path}")
bronze_gate_df = spark.read.format("delta").load(bronze_gate_path)
print(f"Bronze gate_events total rows: {bronze_gate_df.count()}")
bronze_gate_df.show(5, truncate=False)


## Part B — Baggage Scans Raw Ingestion

In [ ]:
# ── Raw Ingestion: baggage_scans.csv ──────────────────────────────
raw_baggage_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_DIR}/baggage_scans/baggage_scans.csv"))

raw_baggage_df = raw_baggage_df.withColumn(
    "ingestion_ts", F.current_timestamp()
)

print(f"Raw baggage scans ingested: {raw_baggage_df.count()}")
raw_baggage_df.printSchema()
raw_baggage_df.show(5, truncate=False)


## Part B — Baggage Scans Data Quality Checks

In [ ]:
# ── DQ Checks: Baggage Scans ──────────────────────────────────────
VALID_SCAN_STATUSES = ["OK", "EXCEPTION", "MISREAD"]

# 1. scan_status validation
invalid_scan_status = raw_baggage_df.filter(
    ~F.col("scan_status").isin(VALID_SCAN_STATUSES)
)
print(f"Invalid scan_status records: {invalid_scan_status.count()}")

# 2. bag_tag_number format validation (should start with BT)
invalid_bag_tag = raw_baggage_df.filter(
    ~F.col("bag_tag_number").rlike("^BT[0-9]+$")
)
print(f"Invalid bag_tag_number format: {invalid_bag_tag.count()}")

# 3. Volume alert: hourly scan count < 5000 → sensor health alert
hourly_counts = (raw_baggage_df
    .withColumn("scan_hour", F.date_trunc("hour", F.to_timestamp("scan_ts")))
    .groupBy("scan_hour").count()
    .filter("count < 5000")
    .orderBy("scan_hour"))
print(f"Hours with < 5,000 scans (sensor health alert): {hourly_counts.count()}")
if hourly_counts.count() > 0:
    hourly_counts.show(truncate=False)

# Build validated dataframe
validated_baggage_df = (raw_baggage_df
    .filter(F.col("scan_status").isin(VALID_SCAN_STATUSES))
    .withColumn("scan_ts", F.to_timestamp("scan_ts"))
    .withColumn("scan_date", F.to_date("scan_ts"))
)
print(f"Validated baggage scans: {validated_baggage_df.count()}")


## Part B — Write Baggage Scans to Bronze Delta

In [ ]:
# ── Write baggage_scans Bronze Delta (append) ────────────────────
bronze_baggage_path = f"{BRONZE_DIR}/baggage_scans"

(validated_baggage_df.write
    .format("delta")
    .mode("append")
    .save(bronze_baggage_path))

print(f"Baggage scans written to {bronze_baggage_path}")
bronze_bag_df = spark.read.format("delta").load(bronze_baggage_path)
print(f"Bronze baggage_scans total rows: {bronze_bag_df.count()}")
bronze_bag_df.show(5, truncate=False)
